## Introduction

The goal is to compare self-published games vs traditionally published games. There are a variety of questions to be answered by this.

For example, do free games tend to be self-published?
What is the price comparison between self-publisehd and published?
What genres are most associated with being self-published? (could make it overall at first, then type in a genre and change it to that genre specifically)
An overlaid line plot comparing self-punlish vs published games

## Set Up

Here is all the code needed to import the necessary libraries for this project. First had to make sure that jupyter was installed into the venv python kernel

In [16]:
import numpy as np

import polars as pl

%matplotlib inline
import matplotlib.pyplot as plt
from sklearn import linear_model as lm

import warnings
warnings.filterwarnings("ignore")

import zipfile
import os

# Plot settings
plt.rcParams['figure.figsize'] = (12, 9)
plt.rcParams['font.size'] = 12

The next step is to read the relevant csv files for this project. Starting with applications.csv which has the most amount of information.

In [21]:
apps = pl.read_csv(
    "steam_dataset_2025_csv/applications.csv",
    try_parse_dates=True,
    infer_schema_length=None,   # scan whole file before deciding dtypes
    null_values=["", "NA", "N/A", "null", "None", "-"],
)

In [23]:
apps.sample(5)

appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,metacritic_score,recommendations_total,mat_supports_windows,mat_supports_mac,mat_supports_linux,mat_initial_price,mat_final_price,mat_discount_percent,mat_currency,mat_achievement_count,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
i64,str,str,bool,date,str,str,str,str,str,f64,f64,bool,bool,bool,f64,f64,f64,str,f64,str,str,str,str,str,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]"
1922770,"""Advanced Chess/Traditional""","""game""",false,2022-06-20,"""0""","""If “chess is the gymnasium of …","""English""","""https://shared.akamai.steamsta…","""https://store.akamai.steamstat…",null,null,true,true,true,499.0,499.0,0.0,"""USD""",null,"""Windows 10 OR Later""",null,"""150 MB RAM""",null,null,null,null,null,2025-09-07 12:36:43.243384 UTC,2025-09-29 06:03:47.805145 UTC
566880,"""Henry The Hamster Handler VR""","""game""",false,2017-01-16,"""0""","""You are Henry, the legendary o…","""English<strong>*</strong>, Sim…","""https://shared.akamai.steamsta…","""https://store.akamai.steamstat…",null,null,true,true,true,99.0,99.0,0.0,"""USD""",29.0,"""Windows 7 SP1, Windows 8.1 or …","""Intel i5-4590 equivalent or be…","""2 GB RAM""","""Nvidia GeForce GTX 970, AMD Ra…","""Windows 7 SP1, Windows 8.1 or …","""Intel i5-4590 equivalent or be…","""4 GB RAM""","""Nvidia GeForce GTX 970, AMD Ra…",2025-09-07 12:36:43.243384 UTC,2025-09-29 06:04:58.894995 UTC
2288320,"""DCS: DCS: AV-8B Kerman Campaig…","""dlc""",false,2023-03-10,"""0""","""Join ‘Detachment B’ of US Mari…","""English<strong>*</strong><br><…","""https://shared.akamai.steamsta…","""https://store.akamai.steamstat…",null,null,true,true,true,1399.0,1399.0,0.0,"""USD""",null,"""Windows 10""",null,null,null,"""Windows 10""",null,null,null,2025-09-07 12:36:43.243384 UTC,2025-09-29 06:03:27.254083 UTC
3811520,"""ZeLastTank : Battle Royale""","""game""",false,2025-07-09,"""0""","""ZeLastTank is a 2D online mult…","""English, French, Italian, Germ…","""https://shared.akamai.steamsta…","""https://store.akamai.steamstat…",null,null,true,true,true,699.0,699.0,0.0,"""USD""",null,"""Windows 10""","""Intel Core i5-2400 / AMD FX-63…","""4 GB RAM""","""NVIDIA GTX 750 Ti / AMD Radeon…","""Windows 10/11 64-bit""","""Intel Core i5-6600K / AMD Ryze…","""16 GB RAM""","""NVIDIA GeForce GTX 1060 / AMD …",2025-09-07 12:36:43.243384 UTC,2025-09-29 06:03:01.355980 UTC
2081810,"""Orion's End""","""game""",false,null,"""0""","""Orion’s End is space trading l…","""English""","""https://shared.akamai.steamsta…","""https://store.akamai.steamstat…",null,null,true,true,true,null,null,null,null,null,"""Windows 10""","""Intel(R) Core(TM) i5-7300U CPU…","""4 GB RAM""","""Intel (R) HD Graphics 620""",null,null,null,null,2025-09-07 12:36:43.243384 UTC,2025-09-29 06:01:56.225423 UTC


As is, the apps dataframe has many unnecessary columns. It also is missing key information about developers, genres, publishers, etc.

I'll have to read the remaining files, and join based on appid. Then, I'll have to make more joins based on publisher id and so on.

Trying to change the respository right now